In [2]:
from pathlib import Path
import json
import pandas as pd

INPUT_DIR = Path("/Users/yuki/csv/weeks2_3_output")
OUTPUT_DIR = Path("/Users/yuki/csv/weeks4_5_output")

DATASET_NAMES = ["sold", "listing"]
SOURCE_ROW = "source_csv_row"

DATE_FIELDS = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
]

NUMERIC_FIELDS = [
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "LotSizeAcres",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "BathroomsFull",
    "BathroomsHalf",
    "BathroomsThreeQuarter",
    "BathroomsOneQuarter",
    "DaysOnMarket",
    "YearBuilt",
    "Latitude",
    "Longitude",
]

PROTECTED_FIELDS = set(DATE_FIELDS + NUMERIC_FIELDS + [
    "ListingId",
    "ListingKey",
    "City",
    "CountyOrParish",
    "PostalCode",
    "StateOrProvince",
    "StandardStatus",
    "PropertySubType",
])

# Add only manually reviewed, unnecessary columns.
EXTRA_COLUMNS_TO_DROP = []

# Approximate California bounding box for preliminary screening.
CA_LAT_MIN, CA_LAT_MAX = 32.5, 42.1
CA_LON_MIN, CA_LON_MAX = -124.5, -114.1

print(f"Pandas version: {pd.__version__}")
print(f"Input folder: {INPUT_DIR}")
print(f"Output folder: {OUTPUT_DIR}")

Pandas version: 2.3.3
Input folder: /Users/yuki/csv/weeks2_3_output
Output folder: /Users/yuki/csv/weeks4_5_output


In [3]:
originals = {}
datasets = {}
metadata = {}
flag_names = {}
audit_parts = {}
transformation_logs = {}


def add_flag(kind, name, condition):
    """Store a boolean flag and register it for reporting."""
    frame = datasets[kind]
    if name in frame.columns:
        raise ValueError(
            f"{name} already exists. Restart from Step 2 before rerunning."
        )
    frame[name] = condition.fillna(False).astype(bool)
    flag_names[kind].append(name)


def record_invalid(kind, field, condition, reason):
    """Preserve original values before replacing invalid data."""
    selected = condition.fillna(False)

    if not selected.any():
        return

    frame = datasets[kind]
    original = originals[kind]

    audit_parts[kind].append(pd.DataFrame({
        SOURCE_ROW: frame.loc[selected, SOURCE_ROW],
        "field": field,
        "original_value": original.loc[selected, field],
        "reason": reason,
    }))


for kind in DATASET_NAMES:
    input_path = INPUT_DIR / kind / "filtered_residential.csv"

    if not input_path.is_file():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    original = pd.read_csv(
        input_path,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    if SOURCE_ROW in original.columns:
        raise ValueError(f"Reserved column already exists: {SOURCE_ROW}")

    # Normalize only blank and whitespace-only values.
    frame = original.replace(r"^\s*$", pd.NA, regex=True).copy()

    # This is the logical CSV record number, including the header.
    frame.insert(0, SOURCE_ROW, range(2, len(frame) + 2))

    originals[kind] = original
    datasets[kind] = frame
    flag_names[kind] = []
    audit_parts[kind] = []

    metadata[kind] = {
        "input_file": str(input_path),
        "rows_before": len(original),
        "columns_before": len(original.columns),
        "missing_before": (
            frame[original.columns].isna().sum().to_dict()
        ),
    }

    transformation_logs[kind] = [
        "Read all original fields as strings to preserve identifiers.",
        "Converted blank and whitespace-only values to missing values.",
        "Preserved other textual missing-value codes for review.",
        "Added source_csv_row to trace records to the input CSV.",
    ]

    print(f"\nDataset: {kind}")
    print(f"Rows: {len(frame):,}")
    print(f"Original columns: {len(original.columns):,}")

    display(frame.head(3))

    display(
        frame[original.columns]
        .isna()
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .rename("missing_count")
        .to_frame()
    )


Dataset: sold
Rows: 414,054
Original columns: 84


,source_csv_row,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,...,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,BuyerAgentAOR,ListAgentAOR,OriginatingSystemName,OriginatingSystemSubName
0,2,"Carpet,Tile,Wood",True,<NA>,<NA>,False,499000.0,551985747,jwachter@cbnorcal.com,2024-01-26,...,94401,6472.0,<NA>,<NA>,True,True,<NA>,<NA>,<NA>,<NA>
1,3,<NA>,False,<NA>,<NA>,False,759900.0,522107581,mdarwich12@gmail.com,2024-01-05,...,91950,<NA>,<NA>,<NA>,True,True,<NA>,<NA>,<NA>,<NA>
2,4,<NA>,False,<NA>,<NA>,False,739900.0,510919001,mdarwich12@gmail.com,2024-01-05,...,91950,<NA>,<NA>,<NA>,True,True,<NA>,<NA>,<NA>,<NA>


,missing_count
AboveGradeFinishedArea,414054
CoveredSpaces,414054
TaxAnnualAmount,414054
TaxYear,414054
ElementarySchoolDistrict,414054
FireplacesTotal,414054
MiddleOrJuniorSchoolDistrict,414054
BusinessType,414054
WaterfrontYN,413795
BelowGradeFinishedArea,411652



Dataset: listing
Rows: 547,162
Original columns: 84


,source_csv_row,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1
0,2,1340000.0,1074973329,haleh360@Gmail.com,<NA>,<NA>,Haleh,Dowlatshahi,34.052207,-118.408445,...,<NA>,False,<NA>,<NA>,90067,<NA>,2105.0,177861.0,<NA>,2220 Avenue Of The Stars 2704
1,3,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,<NA>,<NA>,Renee,Chen,33.496363,-117.691677,...,0,False,3.0,Capistrano Unified,92677,<NA>,254.0,5300.0,<NA>,16 Palisades
2,4,3150000.0,1074936537,anader@dppre.com,<NA>,<NA>,Margaret,Nader,34.119345,-118.111254,...,<NA>,<NA>,2.0,<NA>,91108,<NA>,<NA>,9404.0,<NA>,1615 Waverly Road


,missing_count
TaxAnnualAmount,547162
FireplacesTotal,547162
ElementarySchoolDistrict,547162
TaxYear,547162
BusinessType,547162
MiddleOrJuniorSchoolDistrict,547162
CoveredSpaces,547162
AboveGradeFinishedArea,547162
BelowGradeFinishedArea,544075
CoBuyerAgentFirstName,533466


In [4]:
for kind, frame in datasets.items():
    missing_columns = []
    date_report = []

    for field in DATE_FIELDS:
        if field not in frame.columns:
            missing_columns.append(field)

            frame[field] = pd.Series(
                pd.NaT,
                index=frame.index,
                dtype="datetime64[ns, UTC]",
            )

            invalid = pd.Series(False, index=frame.index)

        else:
            source = frame[field].copy()

            # Naive timestamps are assumed to be UTC.
            parsed = pd.to_datetime(
                source,
                errors="coerce",
                format="mixed",
                utc=True,
            )

            invalid = source.notna() & parsed.isna()

            record_invalid(
                kind, field, invalid, "Unparseable date"
            )

            frame[field] = parsed

        add_flag(kind, f"{field}_parse_error_flag", invalid)
        add_flag(kind, f"{field}_missing_flag", frame[field].isna())

        date_report.append({
            "field": field,
            "dtype": str(frame[field].dtype),
            "parse_errors": int(invalid.sum()),
            "missing_after": int(frame[field].isna().sum()),
        })

    metadata[kind]["missing_date_columns"] = missing_columns

    transformation_logs[kind].append(
        "Parsed four date fields in UTC; unparseable values became NaT. "
        "Absent date columns were added as missing and reported. "
        "Naive timestamps were assumed to be UTC."
    )

    print(f"\nDate conversion: {kind}")
    display(pd.DataFrame(date_report))


Date conversion: sold


,field,dtype,parse_errors,missing_after
0,CloseDate,"datetime64[ns, UTC]",0,0
1,PurchaseContractDate,"datetime64[ns, UTC]",0,194
2,ListingContractDate,"datetime64[ns, UTC]",0,1
3,ContractStatusChangeDate,"datetime64[ns, UTC]",0,594



Date conversion: listing


,field,dtype,parse_errors,missing_after
0,CloseDate,"datetime64[ns, UTC]",0,388096
1,PurchaseContractDate,"datetime64[ns, UTC]",0,285676
2,ListingContractDate,"datetime64[ns, UTC]",0,0
3,ContractStatusChangeDate,"datetime64[ns, UTC]",0,6311


In [5]:
for kind, frame in datasets.items():
    unavailable = [
        field for field in NUMERIC_FIELDS
        if field not in frame.columns
    ]
    metadata[kind]["unavailable_numeric_fields"] = unavailable

    numeric_report = []

    for field in NUMERIC_FIELDS:
        if field not in frame.columns:
            continue

        source = frame[field].copy()

        text = (
            source.astype("string")
            .str.strip()
            .str.replace(r"[$,]", "", regex=True)
        )

        values = pd.to_numeric(text, errors="coerce")
        values = values.replace(
            [float("inf"), float("-inf")], float("nan")
        ).astype("Float64")

        parse_error = source.notna() & values.isna()

        add_flag(kind, f"{field}_parse_error_flag", parse_error)

        record_invalid(
            kind,
            field,
            parse_error,
            "Nonnumeric or nonfinite value",
        )

        frame[field] = values

        invalid = pd.Series(False, index=frame.index)
        rule = None

        if field in ["ClosePrice", "LivingArea"]:
            invalid = values.le(0).fillna(False)
            rule = "Value must be greater than zero"

        elif (
            field in ["DaysOnMarket", "BedroomsTotal"]
            or field.startswith("Bathrooms")
        ):
            invalid = values.lt(0).fillna(False)
            rule = "Value must not be negative"

        if rule is not None:
            add_flag(kind, f"{field}_invalid_value_flag", invalid)
            record_invalid(kind, field, invalid, rule)

            # Keep the row, but exclude invalid values from calculations.
            frame.loc[invalid, field] = pd.NA

        numeric_report.append({
            "field": field,
            "dtype": str(frame[field].dtype),
            "parse_errors": int(parse_error.sum()),
            "invalid_values": int(invalid.sum()),
            "missing_after": int(frame[field].isna().sum()),
        })

    transformation_logs[kind].append(
        "Removed whitespace, dollar signs, and commas from numeric fields. "
        "Converted values to Float64; nonnumeric/nonfinite values became null. "
        "Flagged and nulled nonpositive ClosePrice/LivingArea and negative "
        "DaysOnMarket/bedroom/bathroom values. Retained all rows."
    )

    print(f"\nNumeric conversion: {kind}")
    display(pd.DataFrame(numeric_report))
    print(f"Unavailable numeric fields: {unavailable}")


Numeric conversion: sold


,field,dtype,parse_errors,invalid_values,missing_after
0,ClosePrice,Float64,0,1,3
1,ListPrice,Float64,0,0,0
2,OriginalListPrice,Float64,0,0,756
3,LivingArea,Float64,0,150,384
4,LotSizeAcres,Float64,0,0,32535
5,BedroomsTotal,Float64,0,0,11
6,BathroomsTotalInteger,Float64,0,0,69
7,DaysOnMarket,Float64,0,48,48
8,YearBuilt,Float64,0,0,372
9,Latitude,Float64,0,0,4104


Unavailable numeric fields: ['BathroomsFull', 'BathroomsHalf', 'BathroomsThreeQuarter', 'BathroomsOneQuarter']

Numeric conversion: listing


,field,dtype,parse_errors,invalid_values,missing_after
0,ClosePrice,Float64,0,0,409481
1,ListPrice,Float64,0,0,0
2,OriginalListPrice,Float64,0,0,789
3,LivingArea,Float64,0,360,912
4,LotSizeAcres,Float64,0,0,45030
5,BedroomsTotal,Float64,0,0,133
6,BathroomsTotalInteger,Float64,0,0,55
7,DaysOnMarket,Float64,0,27,27
8,YearBuilt,Float64,0,0,927
9,Latitude,Float64,0,0,80247


Unavailable numeric fields: ['BathroomsFull', 'BathroomsHalf', 'BathroomsThreeQuarter', 'BathroomsOneQuarter']


In [6]:
for kind, frame in datasets.items():
    listing = frame["ListingContractDate"]
    purchase = frame["PurchaseContractDate"]
    close = frame["CloseDate"]

    listing_close_known = listing.notna() & close.notna()
    purchase_close_known = purchase.notna() & close.notna()
    listing_purchase_known = listing.notna() & purchase.notna()

    add_flag(
        kind,
        "listing_after_close_flag",
        listing_close_known & (listing > close),
    )

    add_flag(
        kind,
        "purchase_after_close_flag",
        purchase_close_known & (purchase > close),
    )

    add_flag(
        kind,
        "purchase_before_listing_flag",
        listing_purchase_known & (purchase < listing),
    )

    # Any observed violation of the expected date order.
    add_flag(
        kind,
        "negative_timeline_flag",
        frame["listing_after_close_flag"]
        | frame["purchase_after_close_flag"]
        | frame["purchase_before_listing_flag"],
    )

    add_flag(
        kind,
        "timeline_incomplete_flag",
        listing.isna() | purchase.isna() | close.isna(),
    )

    metadata[kind]["date_comparison_counts"] = {
        "listing_close": int(listing_close_known.sum()),
        "purchase_close": int(purchase_close_known.sum()),
        "listing_purchase": int(listing_purchase_known.sum()),
        "complete_timeline": int(
            (listing.notna() & purchase.notna() & close.notna()).sum()
        ),
    }

    transformation_logs[kind].append(
        "Checked ListingContractDate <= PurchaseContractDate <= CloseDate. "
        "Equal dates were allowed. negative_timeline_flag represents any "
        "observed date-order violation. Missing dates were separately "
        "flagged as an incomplete timeline. Original parsed dates retained."
    )

    date_flags = [
        "listing_after_close_flag",
        "purchase_after_close_flag",
        "purchase_before_listing_flag",
        "negative_timeline_flag",
        "timeline_incomplete_flag",
    ]

    print(f"\nDate consistency: {kind}")
    display(
        frame[date_flags].sum().rename("count").to_frame()
    )


Date consistency: sold


,count
listing_after_close_flag,61
purchase_after_close_flag,239
purchase_before_listing_flag,268
negative_timeline_flag,507
timeline_incomplete_flag,195



Date consistency: listing


,count
listing_after_close_flag,73
purchase_after_close_flag,262
purchase_before_listing_flag,269
negative_timeline_flag,530
timeline_incomplete_flag,388127


In [7]:
geographic_flags = [
    "missing_coordinates_flag",
    "zero_coordinates_flag",
    "positive_longitude_flag",
    "implausible_coordinates_flag",
    "outside_ca_bbox_flag",
]

for kind, frame in datasets.items():
    missing_columns = []

    for field in ["Latitude", "Longitude"]:
        if field not in frame.columns:
            missing_columns.append(field)
            frame[field] = pd.Series(
                pd.NA, index=frame.index, dtype="Float64"
            )

    metadata[kind]["missing_coordinate_columns"] = missing_columns

    latitude = frame["Latitude"].copy()
    longitude = frame["Longitude"].copy()
    complete = latitude.notna() & longitude.notna()

    add_flag(
        kind,
        "missing_coordinates_flag",
        latitude.isna() | longitude.isna(),
    )

    add_flag(
        kind,
        "zero_coordinates_flag",
        (latitude == 0) | (longitude == 0),
    )

    add_flag(
        kind,
        "positive_longitude_flag",
        longitude > 0,
    )

    add_flag(
        kind,
        "implausible_coordinates_flag",
        (latitude < -90) | (latitude > 90)
        | (longitude < -180) | (longitude > 180),
    )

    inside_box = (
        latitude.between(CA_LAT_MIN, CA_LAT_MAX)
        & longitude.between(CA_LON_MIN, CA_LON_MAX)
    )

    add_flag(
        kind,
        "outside_ca_bbox_flag",
        complete & ~inside_box,
    )

    add_flag(
        kind,
        "geographic_review_flag",
        frame[geographic_flags].any(axis=1),
    )

    invalid_latitude = (
        (latitude == 0) | (latitude < -90) | (latitude > 90)
    ).fillna(False)

    invalid_longitude = (
        (longitude == 0) | (longitude > 0)
        | (longitude < -180) | (longitude > 180)
    ).fillna(False)

    for field, invalid in [
        ("Latitude", invalid_latitude),
        ("Longitude", invalid_longitude),
    ]:
        if field in originals[kind].columns:
            record_invalid(
                kind, field, invalid, "Invalid California coordinate"
            )

        frame.loc[invalid, field] = pd.NA

    add_flag(
        kind,
        "coordinates_missing_after_cleaning_flag",
        frame["Latitude"].isna() | frame["Longitude"].isna(),
    )

    transformation_logs[kind].append(
        "Flagged missing, zero, positive-longitude, impossible, and "
        "outside-California-bounding-box coordinates. "
        "Nulled zero/out-of-range latitude and zero/positive/out-of-range "
        "longitude. Retained plausible outside-box values for review. "
        "The bounding box is a screening method, not a state-boundary test."
    )

    print(f"\nGeographic quality: {kind}")
    display(
        frame[
            geographic_flags + [
                "geographic_review_flag",
                "coordinates_missing_after_cleaning_flag",
            ]
        ].sum().rename("count").to_frame()
    )


Geographic quality: sold


,count
missing_coordinates_flag,4104
zero_coordinates_flag,27
positive_longitude_flag,31
implausible_coordinates_flag,2
outside_ca_bbox_flag,92
geographic_review_flag,4196
coordinates_missing_after_cleaning_flag,4163



Geographic quality: listing


,count
missing_coordinates_flag,80247
zero_coordinates_flag,64
positive_longitude_flag,73
implausible_coordinates_flag,4
outside_ca_bbox_flag,312
geographic_review_flag,80559
coordinates_missing_after_cleaning_flag,80385


In [9]:
for kind, frame in datasets.items():
    folder = OUTPUT_DIR / kind
    folder.mkdir(parents=True, exist_ok=True)

    frame.to_csv(
        folder / "cleaned_residential.csv",
        index=False,
        encoding="utf-8-sig",
    )

    dtype_reports[kind].to_csv(
        folder / "dtype_report.csv", index=False
    )

    missing_reports[kind].to_csv(
        folder / "missing_value_report.csv", index=False
    )

    flag_reports[kind].to_csv(
        folder / "flag_counts.csv", index=False
    )

    dropped_reports[kind].to_csv(
        folder / "dropped_columns.csv", index=False
    )

    audit = (
        pd.concat(audit_parts[kind], ignore_index=True)
        if audit_parts[kind]
        else pd.DataFrame(columns=[
            SOURCE_ROW, "field", "original_value", "reason"
        ])
    )

    audit.to_csv(
        folder / "invalid_value_audit.csv", index=False
    )

    geographic_review = frame.loc[
        frame["geographic_review_flag"]
    ].copy()

    for field in ["Latitude", "Longitude"]:
        if field in originals[kind].columns:
            geographic_review[f"original_{field}"] = (
                originals[kind].loc[geographic_review.index, field]
            )

    geographic_review.to_csv(
        folder / "geographic_review_records.csv", index=False
    )

    geographic_summary = {
        field: int(frame[field].sum())
        for field in geographic_flags + [
            "geographic_review_flag",
            "coordinates_missing_after_cleaning_flag",
        ]
    }

    summary = {
        **metadata[kind],
        "flag_counts": {
            field: int(frame[field].sum())
            for field in flag_names[kind]
        },
        "geographic_summary": geographic_summary,
        "geographic_method": (
            "Approximate California bounding-box screening only."
        ),
        "california_bounding_box": {
            "latitude_min": CA_LAT_MIN,
            "latitude_max": CA_LAT_MAX,
            "longitude_min": CA_LON_MIN,
            "longitude_max": CA_LON_MAX,
        },
        "negative_timeline_definition": (
            "Any observed violation of ListingContractDate <= "
            "PurchaseContractDate <= CloseDate, including listing after close."
        ),
        "transformations": transformation_logs[kind],
        "notes": [
            "Flag counts overlap; do not sum them as unique records.",
            "Missing closing information may be expected for listings.",
            "Use relevant flags to select records for each analysis.",
            "CSV does not preserve Pandas data types.",
        ],
    }

    (folder / "cleaning_summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    notes = [
        f"Dataset: {kind}",
        f"Input file: {metadata[kind]['input_file']}",
        f"Rows before: {metadata[kind]['rows_before']:,}",
        f"Rows after: {metadata[kind]['rows_after']:,}",
        f"Rows removed: {metadata[kind]['rows_removed']:,}",
        "",
        "Transformations:",
        *[
            f"{number}. {text}"
            for number, text in enumerate(
                transformation_logs[kind], start=1
            )
        ],
        "",
        "Flag counts:",
        flag_reports[kind].to_string(index=False),
        "",
        "Geographic quality:",
        *[
            f"{field}: {count:,}"
            for field, count in geographic_summary.items()
        ],
        "",
        "Coordinate screening uses an approximate bounding box.",
        "Flag counts overlap and must not be summed as unique records.",
        "CSV does not preserve Pandas data types.",
    ]

    (folder / "cleaning_summary.txt").write_text(
        "\n".join(notes) + "\n",
        encoding="utf-8",
    )

    print(f"\nSaved {kind} outputs to: {folder}")
    print(f"Cleaned dataset: {folder / 'cleaned_residential.csv'}")

print("\nAll exports completed.")


Saved sold outputs to: /Users/yuki/csv/weeks4_5_output/sold
Cleaned dataset: /Users/yuki/csv/weeks4_5_output/sold/cleaned_residential.csv

Saved listing outputs to: /Users/yuki/csv/weeks4_5_output/listing
Cleaned dataset: /Users/yuki/csv/weeks4_5_output/listing/cleaned_residential.csv

All exports completed.
